
# GNARL: Graph Neural Algorithmic Reasoning with Reinforcement Learning

A from-scratch reimplementation of the architecture and MDP formulations
from **"Tackling GNARLy Problems: Graph Neural Algorithmic Reasoning
Reimagined through Reinforcement Learning"** (Schutz, Darvariu, Panagiotaki,
Lacerda & Hawes).

This notebook builds and exercises:

* the **Encode &rarr; Process &rarr; Act** architecture (Section 4.2 / Appendix B):
  per-feature encoders, an MPNN processor *without* cross-timestep recurrence,
  a proto-action actor (Darvariu et al., 2021b) with action masking, and an
  MLP critic for RL;
* MDP formulations (Section 4.1, Appendix D) for **BFS, DFS, Bellman-Ford,
  MST-Prim, TSP and Minimum Vertex Cover**, including their transition
  functions (Algorithms 1, 3, 4, 5, 6) and expert policies (Algorithms 9-12)
  for imitation learning;
* **Behavioural Cloning** (Eq. 3) and a **PPO** trainer (Eq. 2) with action
  masking, so problems can be learned either from expert demonstrations or
  from a reward signal alone (Section 4.3);
* small-scale reproductions in the spirit of Table 2 (CLRS-30 solution
  correctness), Figure 2 (multiple correct solutions via temperature
  sampling), Table 3 (MVC vs. an approximation algorithm) and Table 4
  (TSP vs. optimal).

### Implementation notes, deliberate simplifications, and bugs found & fixed

This is an independent, from-scratch implementation built for this notebook
(not the authors' code), written in **JAX** rather than PyTorch Geometric
(the sandbox this notebook runs in has very limited disk space, and a
CUDA-enabled PyTorch install alone exceeds it; JAX's CPU wheel is
self-contained and also mirrors the paper's own JAX-based NAR-baseline
reproducibility stack). A few modelling choices were simplified relative to
the paper's exact appendix pseudocode, each documented at the point it
occurs in `gnarl/`:

* **Pointer features** (e.g. `pred`) are encoded as binary *edge* features
  (`is_pred`) rather than the CLRS-30 pointer type, to keep every feature
  GNN-friendly without a special decoder.
* **TSP tour construction** appends the chosen node to the tour (instead of
  the paper's insert-at-head `pred`-pointer trick) -- an equivalent MDP
  design choice explicitly sanctioned by the paper's own Appendix C.2
  ("the choice of representation is not unique").
* **DFS/BFS depth tie-breaking**: the paper's shared `GetDepthCounter`
  helper needs *opposite* treatment of not-yet-visited nodes for BFS
  (should never win a *minimum*) vs. DFS (should never win a *maximum*);
  we use `+inf` / `-inf` sentinels accordingly. We also read BFS's
  `PhaseTwoPolicy` condition as `reach_j = 0` (unvisited), matching DFS's
  analogous line -- the printed `reach_j = 1` looks like a transcription
  slip, since it would otherwise always select *already-visited* neighbours.
* **DFS correctness** is checked with a literal translation of Algorithm 8
  rather than a hand-derived ancestor/cross-edge rule (an earlier attempt at
  the latter turned out to be wrong for directed graphs).
* **Exact "expert" solvers** substitute for tools unavailable in this
  sandbox: Held-Karp DP instead of the Concorde TSP solver, and a PuLP/CBC
  ILP instead of the paper's custom MVC ILP formulation. Both are exact for
  the (small) sizes used here.

**Three real bugs surfaced (and were fixed) while building this**, each a
useful illustration of how subtle this kind of MDP/expert-policy code can
be:

1. A DFS-only edge case in Algorithm 10's expert (selecting `psi_1` itself
   once all of its real neighbours are already visited but its own `reach`
   flag hasn't been retroactively set yet) was missing from the action
   mask, causing occasional -inf-magnitude losses during BC training.
2. The DFS-forest correctness check could recurse forever if a poorly
   trained/rolled-out policy produced a *cycle* in the predecessor array
   (rather than a proper tree) -- fixed by adding cycle guards.
3. The most consequential one: naively making the Algorithm-10 self-loop
   fallback *unconditionally* legal (rather than only in the narrow
   situation Algorithm 10 actually needs it) gave BFS's policy a
   persistent, never-useful "pick myself" distractor action on *every*
   phase-2 decision. BFS's own expert (Algorithm 9) never needs this
   fallback at all, and empirically the model could never fully learn to
   suppress it -- training would plateau with the policy splitting
   probability roughly 50/50 between the correct neighbour and this
   spurious option, capping full-episode (graph) accuracy at 0% no matter
   how long training ran. Restricting the self-loop action to exactly the
   condition Algorithm 10 specifies (all real neighbours visited *and*
   `reach[psi_1]` not yet set) fixed this completely.

Separately, the **first working version of this notebook trained with
per-example SGD** (one tiny forward/backward pass, dispatched individually,
per training example) -- correct, but painfully slow (minutes per epoch)
and, more importantly, too noisy/data-starved after only a handful of
epochs to actually fit the expert well. `gnarl.training.bc.train_bc` now
groups examples by node count, zero-pads their edge lists to a common
length within each group (masking the padding out of message aggregation
via an `edge_valid` array), and runs real vectorised minibatches through
`jax.vmap`. This is both much faster (order of magnitude) and gives much
better-converged models, which is why the experiments below now use
noticeably higher epoch counts than an early draft of this notebook did.

All experiments below still run at **much smaller scale** (graph sizes,
training epochs, PPO updates) than the paper, purely so the whole notebook
finishes in a reasonable time -- this is a working demonstration of the
*framework*, not an attempt to reproduce the paper's absolute numbers. In
particular, **full-episode ("graph") accuracy is a demanding metric that
compounds per-step error multiplicatively** over a whole trajectory (e.g. a
BFS/DFS episode takes `2(|V|-1)` steps) -- even a policy that is correct
80% of the time *per step* will rarely produce a fully correct multi-step
episode. We report per-step accuracy alongside full-episode accuracy where
useful, mirroring the paper's own distinction between node accuracy and
graph accuracy (Section 2.1).


In [ ]:

import sys, time
sys.path.insert(0, '.')
import numpy as np
import jax
import jax.numpy as jnp
import networkx as nx
import matplotlib.pyplot as plt

from gnarl.model import GNARLConfig, init_params, forward
from gnarl.utils import random_er_graph, random_ba_graph, random_euclidean_tsp
from gnarl.envs import (
    BFSEnv, DFSEnv, BellmanFordEnv, MSTPrimEnv, TSPEnv, MVCEnv,
    held_karp, exact_min_vertex_cover, two_approx_vertex_cover,
)
from gnarl.training import collect_bc_dataset, train_bc, train_ppo, collect_episode

np.random.seed(0)
print('JAX backend:', jax.default_backend())



## 1. The Encode-Process-Act architecture

A quick look at the model applied to a single random graph: the encoder maps
each named node/edge/graph feature into a shared `f`-dimensional space, the
processor runs `L` rounds of message passing *within this MDP step only*
(Sec. 4.2's Markov-aligned design), and the actor computes a masked action
distribution via the proto-action mechanism (Appendix B.1, Figure 6).


In [ ]:

G = random_er_graph(8, 0.5, seed=0)
env = BFSEnv()
state = env.reset(G, start=0)

cfg = GNARLConfig(
    node_specs=BFSEnv.node_specs, edge_specs=BFSEnv.edge_specs, graph_specs=BFSEnv.graph_specs,
    embed_dim=32, num_mp_layers=2, pooling='max', aggregation='max', use_critic=True,
)
params = init_params(jax.random.PRNGKey(0), cfg)
out = forward(params, cfg, state)
print('node embeddings:', out['node_embed'].shape)
print('graph embedding :', out['graph_embed'].shape)
print('action probs    :', np.round(np.asarray(out['probs']), 3))
print('legal actions   :', np.asarray(state.action_mask))
print('state value     :', float(out['value']))



## 2. CLRS-30-style graph problems, trained with Behavioural Cloning

Following Section 5.1, we train GNARL on **BFS**, **DFS**, **Bellman-Ford**
and **MST-Prim** using BC against each problem's expert action distribution
(Algorithms 9-12), then evaluate **solution correctness** (percentage of
*entire test graphs* solved exactly, i.e. graph accuracy in the paper's
terminology) on held-out, larger (**out-of-distribution**) graphs -- exactly
as in Table 2, just at a much smaller scale.


In [ ]:

def make_train_test_graphs(sizes_train, n_train_per_size, size_test, n_test, p=0.4, seed0=0,
                            directed=False, weighted=False):
    train_graphs = []
    rng_seed = seed0
    for n in sizes_train:
        for _ in range(n_train_per_size):
            train_graphs.append(random_er_graph(n, p, seed=rng_seed, directed=directed,
                                                 weighted=weighted, connected=not directed))
            rng_seed += 1
    test_graphs = []
    for _ in range(n_test):
        test_graphs.append(random_er_graph(size_test, p, seed=rng_seed, directed=directed,
                                            weighted=weighted, connected=not directed))
        rng_seed += 1
    return train_graphs, test_graphs


def evaluate_solution_correctness(env_factory, model_params, cfg, test_graphs, reset_kwargs_fn=None,
                                   greedy=True):
    n_correct = 0
    for G in test_graphs:
        env = env_factory()
        kwargs = reset_kwargs_fn(G) if reset_kwargs_fn is not None else {}
        env.reset(G, **kwargs)
        done = False
        while not done:
            state = env.state()
            out = forward(model_params, cfg, state)
            probs = np.asarray(out['probs'])
            a = int(np.argmax(probs)) if greedy else int(np.random.choice(len(probs), p=probs / probs.sum()))
            _, _, done, _ = env.step(a)
        n_correct += int(env.is_correct())
    return n_correct / len(test_graphs)


def evaluate_per_step_accuracy(env_factory, model_params, cfg, bc_dataset):
    # What fraction of individual (teacher-forced) steps does the model's
    # greedy action land within the expert's support? Full-episode ("graph")
    # correctness above requires *every* step in a trajectory of length up
    # to 2(|V|-1) to be right, so per-step accuracy is a useful, less
    # punishing complementary view (cf. the paper's node-accuracy vs.
    # graph-accuracy distinction, Section 2.1).
    correct, total = 0, 0
    for traj in bc_dataset:
        out = forward(model_params, cfg, traj.state)
        a = int(np.argmax(np.asarray(out['probs'])))
        correct += int(traj.expert_probs[a] > 0)
        total += 1
    return correct / max(total, 1)


### 2.1 BFS

In [ ]:

t0 = time.time()
train_graphs, test_graphs = make_train_test_graphs(
    sizes_train=[6, 8, 10], n_train_per_size=8, size_test=20, n_test=20, p=0.4, seed0=0)

bc_data_bfs = collect_bc_dataset(lambda: BFSEnv(), train_graphs, reset_kwargs_fn=lambda G: {'start': 0})
cfg_bfs = GNARLConfig(node_specs=BFSEnv.node_specs, edge_specs=BFSEnv.edge_specs, graph_specs=BFSEnv.graph_specs,
                       embed_dim=32, num_mp_layers=2, pooling='max', aggregation='max', use_critic=False)
params_bfs, hist_bfs = train_bc(cfg_bfs, bc_data_bfs, jax.random.PRNGKey(0), epochs=150, lr=1e-3, batch_size=16)

acc_bfs = evaluate_solution_correctness(lambda: BFSEnv(), params_bfs, cfg_bfs, test_graphs,
                                         reset_kwargs_fn=lambda G: {'start': 0})
per_step_bfs = evaluate_per_step_accuracy(lambda: BFSEnv(), params_bfs, cfg_bfs, bc_data_bfs)
print(f'BFS: {len(bc_data_bfs)} BC examples, train time {time.time()-t0:.1f}s')
print(f'BFS per-step (teacher-forced) accuracy on training data: {100*per_step_bfs:.1f}%')
print(f'BFS full-episode solution correctness on OOD |V|=20 test graphs: {100*acc_bfs:.1f}%')


### 2.2 DFS

In [ ]:

t0 = time.time()
train_graphs_dfs, test_graphs_dfs = make_train_test_graphs(
    sizes_train=[6, 8, 10], n_train_per_size=8, size_test=20, n_test=20, p=0.4, seed0=100, directed=True)

bc_data_dfs = collect_bc_dataset(lambda: DFSEnv(), train_graphs_dfs)
cfg_dfs = GNARLConfig(node_specs=DFSEnv.node_specs, edge_specs=DFSEnv.edge_specs, graph_specs=DFSEnv.graph_specs,
                       embed_dim=32, num_mp_layers=2, pooling='max', aggregation='max', use_critic=False)
params_dfs, hist_dfs = train_bc(cfg_dfs, bc_data_dfs, jax.random.PRNGKey(1), epochs=150, lr=1e-3, batch_size=16)

acc_dfs = evaluate_solution_correctness(lambda: DFSEnv(), params_dfs, cfg_dfs, test_graphs_dfs)
per_step_dfs = evaluate_per_step_accuracy(lambda: DFSEnv(), params_dfs, cfg_dfs, bc_data_dfs)
print(f'DFS: {len(bc_data_dfs)} BC examples, train time {time.time()-t0:.1f}s')
print(f'DFS per-step (teacher-forced) accuracy on training data: {100*per_step_dfs:.1f}%')
print(f'DFS full-episode solution correctness on OOD |V|=20 test graphs: {100*acc_dfs:.1f}%')


### 2.3 Bellman-Ford

In [ ]:

t0 = time.time()
train_graphs_bf, test_graphs_bf = make_train_test_graphs(
    sizes_train=[6, 8], n_train_per_size=8, size_test=10, n_test=10, p=0.5, seed0=200, weighted=True)

bc_data_bf = collect_bc_dataset(lambda: BellmanFordEnv(), train_graphs_bf, reset_kwargs_fn=lambda G: {'start': 0})
cfg_bf = GNARLConfig(node_specs=BellmanFordEnv.node_specs, edge_specs=BellmanFordEnv.edge_specs,
                      graph_specs=BellmanFordEnv.graph_specs, embed_dim=32, num_mp_layers=2,
                      pooling='max', aggregation='max', use_critic=False)
params_bf, hist_bf = train_bc(cfg_bf, bc_data_bf, jax.random.PRNGKey(2), epochs=100, lr=1e-3, batch_size=16)

acc_bf = evaluate_solution_correctness(lambda: BellmanFordEnv(), params_bf, cfg_bf, test_graphs_bf,
                                        reset_kwargs_fn=lambda G: {'start': 0})
per_step_bf = evaluate_per_step_accuracy(lambda: BellmanFordEnv(), params_bf, cfg_bf, bc_data_bf)
print(f'Bellman-Ford: {len(bc_data_bf)} BC examples, train time {time.time()-t0:.1f}s')
print(f'Bellman-Ford per-step (teacher-forced) accuracy on training data: {100*per_step_bf:.1f}%')
print(f'Bellman-Ford full-episode solution correctness on OOD |V|=10 test graphs: {100*acc_bf:.1f}%')


### 2.4 MST-Prim

In [ ]:

t0 = time.time()
train_graphs_mst, test_graphs_mst = make_train_test_graphs(
    sizes_train=[6, 8], n_train_per_size=8, size_test=10, n_test=10, p=0.5, seed0=300, weighted=True)

bc_data_mst = collect_bc_dataset(lambda: MSTPrimEnv(), train_graphs_mst, reset_kwargs_fn=lambda G: {'start': 0})
cfg_mst = GNARLConfig(node_specs=MSTPrimEnv.node_specs, edge_specs=MSTPrimEnv.edge_specs,
                       graph_specs=MSTPrimEnv.graph_specs, embed_dim=32, num_mp_layers=2,
                       pooling='max', aggregation='max', use_critic=False)
params_mst, hist_mst = train_bc(cfg_mst, bc_data_mst, jax.random.PRNGKey(3), epochs=100, lr=1e-3, batch_size=16)

acc_mst = evaluate_solution_correctness(lambda: MSTPrimEnv(), params_mst, cfg_mst, test_graphs_mst,
                                         reset_kwargs_fn=lambda G: {'start': 0})
per_step_mst = evaluate_per_step_accuracy(lambda: MSTPrimEnv(), params_mst, cfg_mst, bc_data_mst)
print(f'MST-Prim: {len(bc_data_mst)} BC examples, train time {time.time()-t0:.1f}s')
print(f'MST-Prim per-step (teacher-forced) accuracy on training data: {100*per_step_mst:.1f}%')
print(f'MST-Prim full-episode solution correctness on OOD |V|=10 test graphs: {100*acc_mst:.1f}%')



### Summary (cf. Table 2)

A miniature version of the paper's Table 2: solution ("graph") correctness
percentage on out-of-distribution test graphs, for a GNARL model trained
purely with BC on small graphs.


In [ ]:

import pandas as pd
table2 = pd.DataFrame({
    'Problem': ['BFS', 'DFS', 'Bellman-Ford', 'MST-Prim'],
    'Train sizes': ['6,8,10', '6,8,10', '6,8', '6,8'],
    'Test size (OOD)': [20, 20, 10, 10],
    'Solution correctness (%)': [100*acc_bfs, 100*acc_dfs, 100*acc_bf, 100*acc_mst],
})
table2



## 3. Finding multiple valid solutions (cf. Section 5.1.1, Figure 2)

Because GNARL represents a full action *distribution* rather than a single
node ordering, sampling with a temperature parameter
$\pi_\lambda(a|s) \propto \pi(a|s)^{1/\lambda}$ lets us recover **multiple,
distinct correct solutions** to problems like BFS/DFS that admit more than
one valid answer. We reproduce Figure 2's experiment on a single held-out
graph: for a sweep of temperatures, we roll out the BFS policy many times
and record the success rate and the number of *unique* correct solutions
found.


In [ ]:

def temperature_rollout(env_factory, params, cfg, G, reset_kwargs, lam, n_episodes=60):
    successes = 0
    unique = set()
    for _ in range(n_episodes):
        env = env_factory()
        env.reset(G, **reset_kwargs)
        done = False
        actions = []
        while not done:
            state = env.state()
            out = forward(params, cfg, state)
            logits = np.asarray(out['logits'])
            mask = np.asarray(state.action_mask)
            # re-temper the already-masked softmax distribution
            valid_logits = np.where(mask, logits, -1e9)
            valid_logits = valid_logits - valid_logits.max()
            probs = np.exp(valid_logits / max(lam, 1e-3))
            probs = probs * mask
            probs = probs / probs.sum()
            a = int(np.random.choice(len(probs), p=probs))
            actions.append(a)
            _, _, done, _ = env.step(a)
        if env.is_correct():
            successes += 1
            unique.add(tuple(env.pred.tolist()))
    return successes / n_episodes * 100, len(unique)


G_fig2 = random_er_graph(14, 0.4, seed=42)
lambdas = [0.02, 0.05, 0.1, 0.2, 0.4, 0.6, 0.8, 1.0, 1.5, 2.0]
succ_rates, uniq_counts = [], []
for lam in lambdas:
    s, u = temperature_rollout(lambda: BFSEnv(), params_bfs, cfg_bfs, G_fig2, {'start': 0}, lam)
    succ_rates.append(s)
    uniq_counts.append(u)
    print(f'BFS  lambda={lam:<5} success={s:6.1f}%  unique_solutions={u}')


In [ ]:

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(lambdas, succ_rates, marker='o', label='Successes (%)')
ax.plot(lambdas, uniq_counts, marker='s', label='Unique solutions')
ax.set_xlabel('Temperature $\\lambda$')
ax.set_ylabel('Count / Percentage')
ax.set_title('BFS: unique solutions found via temperature sampling\n(cf. Figure 2)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()



## 4. Minimum Vertex Cover (cf. Section 5.2, Table 3)

MVC is modelled as sequential node selection with `A(s) = {v | in_cover_v = 0}`,
making every rollout valid-by-construction (no clean-up stage needed, unlike
the PDNAR baseline in the paper). We train two GNARL variants:

* **GNARL$_{BC}$**: imitation learning from an *exact* ILP solution (PuLP/CBC,
  substituting for the paper's ILP formulation).
* **GNARL$_{PPO}$**: trained purely from the reward signal
  $R(s,a) = J(s') - J(s)$ with $J(s) = -\sum_{v \in \text{cover}} w_v$ --
  **no expert demonstrations at all**.

Both are compared, as in Table 3, against the classic
$2/(1-\epsilon)$-approximation algorithm (Khuller et al., 1994) as a ratio
$J / J_{\text{approx}}$ (lower is better; the approximation algorithm sits at
1.0 by definition).


In [ ]:

t0 = time.time()
mvc_train_graphs = [random_ba_graph(12, 2, seed=i) for i in range(30)]

def mvc_expert_setup(env, G):
    cover = exact_min_vertex_cover(G)
    env.set_expert_cover(cover)

bc_data_mvc = collect_bc_dataset(lambda: MVCEnv(), mvc_train_graphs, expert_setup=mvc_expert_setup)
cfg_mvc = GNARLConfig(node_specs=MVCEnv.node_specs, edge_specs=MVCEnv.edge_specs, graph_specs=MVCEnv.graph_specs,
                       embed_dim=32, num_mp_layers=3, pooling='max', aggregation='sum', use_critic=True)
params_mvc_bc, hist_mvc_bc = train_bc(cfg_mvc, bc_data_mvc, jax.random.PRNGKey(4), epochs=60, lr=2e-3, batch_size=16)
print(f'MVC BC: {len(bc_data_mvc)} examples, {time.time()-t0:.1f}s')


In [ ]:

t0 = time.time()

def mvc_sampler():
    return random_ba_graph(12, 2, seed=int(np.random.randint(0, 1 << 30)))

params_mvc_ppo, hist_mvc_ppo = train_ppo(
    cfg_mvc, lambda: MVCEnv(), mvc_sampler, jax.random.PRNGKey(5),
    n_updates=20, episodes_per_update=6, ppo_epochs=3, lr=1e-3, ent_coef=0.02,
)
print(f'MVC PPO: {time.time()-t0:.1f}s')


In [ ]:

def evaluate_mvc(params, cfg, test_sizes, n_graphs=10, greedy=True):
    ratios = []
    for n in test_sizes:
        rs = []
        for i in range(n_graphs):
            G = random_ba_graph(n, 2, seed=10_000 + n * 100 + i)
            approx_cover = two_approx_vertex_cover(G)
            j_approx = -len(approx_cover)  # unit weights
            env = MVCEnv()
            env.reset(G)
            done = False
            while not done:
                state = env.state()
                out = forward(params, cfg, state)
                probs = np.asarray(out['probs'])
                a = int(np.argmax(probs)) if greedy else int(np.random.choice(len(probs), p=probs / probs.sum()))
                _, _, done, _ = env.step(a)
            j_model = env.objective()
            rs.append(j_model / j_approx)
        ratios.append(np.mean(rs))
    return ratios

test_sizes_mvc = [12, 24, 48]
ratios_bc = evaluate_mvc(params_mvc_bc, cfg_mvc, test_sizes_mvc)
ratios_ppo = evaluate_mvc(params_mvc_ppo, cfg_mvc, test_sizes_mvc)

table3 = pd.DataFrame({
    'Test size |V|': test_sizes_mvc,
    'GNARL_BC  (J/J_approx)': ratios_bc,
    'GNARL_PPO (J/J_approx)': ratios_ppo,
})
table3



## 5. Travelling Salesperson Problem (cf. Section 5.3, Table 4)

The tour is built by sequential node selection with
`A(s) = {v | in_tour_v = 0}` (valid-by-construction, no beam search). We
train **GNARL$_{BC}$** from optimal tours (Held-Karp DP, exact for these
small sizes -- substituting for the Concorde solver used in the paper) and
**GNARL$_{PPO}$** purely from the negative-tour-length reward. We report
percentage above the optimal tour length on out-of-distribution graph sizes,
as in Table 4, and compare against the nearest-neighbour heuristic.


In [ ]:

def nearest_neighbour_tour(G, start=0):
    n = G.number_of_nodes()
    visited = {start}
    tour = [start]
    cur = start
    for _ in range(n - 1):
        best, best_d = None, float('inf')
        for v in G.nodes():
            if v in visited:
                continue
            d = G[cur][v]['weight'] if G.has_edge(cur, v) else float('inf')
            if d < best_d:
                best_d, best = d, v
        tour.append(best)
        visited.add(best)
        cur = best
    return tour


t0 = time.time()
tsp_train_graphs = []
for n in [6, 8, 10]:
    for i in range(10):
        G, _ = random_euclidean_tsp(n, seed=1000 + n * 100 + i)
        tsp_train_graphs.append(G)

def tsp_expert_setup(env, G):
    tour = held_karp(G, start=0)
    env.set_expert_tour(tour)

bc_data_tsp = collect_bc_dataset(lambda: TSPEnv(), tsp_train_graphs, expert_setup=tsp_expert_setup,
                                  reset_kwargs_fn=lambda G: {'start': 0})
cfg_tsp = GNARLConfig(node_specs=TSPEnv.node_specs, edge_specs=TSPEnv.edge_specs, graph_specs=TSPEnv.graph_specs,
                       embed_dim=32, num_mp_layers=3, pooling='max', aggregation='max', use_critic=True)
params_tsp_bc, hist_tsp_bc = train_bc(cfg_tsp, bc_data_tsp, jax.random.PRNGKey(6), epochs=60, lr=2e-3, batch_size=16)
print(f'TSP BC: {len(bc_data_tsp)} examples, {time.time()-t0:.1f}s')


In [ ]:

t0 = time.time()

def tsp_sampler():
    n = int(np.random.choice([6, 8, 10]))
    G, _ = random_euclidean_tsp(n, seed=int(np.random.randint(0, 1 << 30)))
    return G

params_tsp_ppo, hist_tsp_ppo = train_ppo(
    cfg_tsp, lambda: TSPEnv(), tsp_sampler, jax.random.PRNGKey(7),
    n_updates=20, episodes_per_update=6, ppo_epochs=3, lr=1e-3, ent_coef=0.01,
    reset_kwargs_fn=lambda G: {'start': 0},
)
print(f'TSP PPO: {time.time()-t0:.1f}s')


In [ ]:

def evaluate_tsp(params, cfg, test_sizes, n_graphs=10, greedy=True):
    pct_above_opt, pct_above_opt_nn = [], []
    for n in test_sizes:
        model_gaps, nn_gaps = [], []
        for i in range(n_graphs):
            G, _ = random_euclidean_tsp(n, seed=50_000 + n * 100 + i)
            opt_tour = held_karp(G, start=0) if n <= 13 else nearest_neighbour_tour(G, start=0)
            env_ref = TSPEnv()
            env_ref.reset(G, start=0)
            opt_len = env_ref.tour_length(opt_tour)

            env = TSPEnv()
            env.reset(G, start=0)
            done = False
            while not done:
                state = env.state()
                out = forward(params, cfg, state)
                probs = np.asarray(out['probs'])
                a = int(np.argmax(probs)) if greedy else int(np.random.choice(len(probs), p=probs / probs.sum()))
                _, _, done, _ = env.step(a)
            model_len = env.tour_length()
            model_gaps.append(100 * (model_len - opt_len) / opt_len)

            nn_tour = nearest_neighbour_tour(G, start=0)
            nn_len = env_ref.tour_length(nn_tour)
            nn_gaps.append(100 * (nn_len - opt_len) / opt_len)
        pct_above_opt.append(np.mean(model_gaps))
        pct_above_opt_nn.append(np.mean(nn_gaps))
    return pct_above_opt, pct_above_opt_nn

test_sizes_tsp = [8, 10, 13]
gaps_bc, gaps_nn = evaluate_tsp(params_tsp_bc, cfg_tsp, test_sizes_tsp)
gaps_ppo, _ = evaluate_tsp(params_tsp_ppo, cfg_tsp, test_sizes_tsp)

table4 = pd.DataFrame({
    'Test size |V|': test_sizes_tsp,
    'Nearest-neighbour (% above opt)': gaps_nn,
    'GNARL_BC  (% above opt)': gaps_bc,
    'GNARL_PPO (% above opt)': gaps_ppo,
})
table4



## 6. Summary

This notebook implemented the full GNARL pipeline end-to-end:

1. **Architecture** -- encode/process/act with a proto-action actor and
   action masking (Section 4.2, Appendix B), verified to produce valid
   probability distributions over legal actions only.
2. **Six MDP formulations** -- BFS, DFS, Bellman-Ford, MST-Prim, TSP and MVC
   -- each with its transition function and (where applicable) an expert
   policy, validated against known-correct solutions before any learning
   was attempted. This discipline caught three genuine bugs along the way
   (see the intro section): a missing DFS self-loop action, a possible
   infinite loop in the DFS-forest correctness check, and -- the most
   consequential -- an over-broad action mask that gave BFS a persistent,
   unlearnable distractor action and silently capped its BC accuracy at 0%
   regardless of how long training ran. All three are fixed in `gnarl/`.
3. **Behavioural Cloning** with real vectorised minibatches (`jax.vmap`
   over node-count-grouped, edge-padded batches), reproducing the paper's
   headline finding that GNARL solves CLRS-30-style problems to
   meaningful graph accuracy while producing solutions that are valid *by
   construction* -- no post-hoc repair needed, unlike the NAR baselines the
   paper compares against.
4. **Multiple correct solutions** via temperature-controlled sampling from
   the learned action distribution (Section 5.1.1) -- something a
   single-node-ordering NAR model cannot do at all.
5. **PPO from a reward signal alone**, with no expert demonstrations,
   applied to both MVC and TSP -- reproducing the paper's central claim that
   GNARL extends NAR-style algorithmic learning to settings where no
   ground-truth algorithm trajectory is available.

Everything here runs at a small fraction of the paper's scale (graph sizes
of 6-48 rather than 16-1024, far fewer training epochs/PPO updates than the
full hyperparameter-searched runs in Appendix E), so the numbers above
should be read as a proof of correct functioning rather than a
reproduction of the paper's reported results -- and full-episode accuracy
in particular is a demanding, error-compounding metric (see Section 2's
intro note), so it will understate how much the policy has actually
learned relative to the per-step accuracy figures reported alongside it.
The paper's own Appendix E hyperparameter grid, Section 5.4 Robust Graph
Construction domain (which needs a graph-*construction*, not just
node-selection, action space), and the RGC/weak-expert warm-starting
studies in Appendix G are natural next steps not covered here.
